# Task 1: Exploratory Data Analysis (EDA) on the Titanic Dataset
**InternCircle Data Science Internship** | **Author:** Vignesh Murugesan

**Objective:** Load and analyze the Titanic dataset with Pandas to find key statistical insights.

**Plan**
1. Load data and inspect it (`head`, `info`, `describe`)
2. Handle missing values and duplicates
3. Feature engineering (title, family size, age group)
4. Grouping and aggregation analysis
5. Key findings

## 1. Load the data

In [1]:
import pandas as pd
import numpy as np

URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(URL)
print("Rows, columns:", df.shape)

Rows, columns: (891, 12)


## 2. DataFrame operations: `head`, `info`, `describe`

In [2]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [4]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [5]:
# Summary of categorical columns
df.describe(include="object")

/tmp/ipykernel_216/292326849.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include="object")


,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,204,889
unique,891,2,681,147,3
top,"Braund, Mr. Owen Harris",male,347082,G6,S
freq,1,577,7,4,644


**Data dictionary (key columns)**

| Column | Meaning |
|---|---|
| Survived | 0 = No, 1 = Yes |
| Pclass | Ticket class (1 = upper, 3 = lower) |
| Sex, Age | Passenger sex and age |
| SibSp / Parch | Siblings-spouses / parents-children aboard |
| Fare | Ticket price |
| Cabin | Cabin number |
| Embarked | Port: C = Cherbourg, Q = Queenstown, S = Southampton |

## 3. Handling missing values and duplicates

In [6]:
missing = df.isna().sum()
pct = (missing / len(df) * 100).round(1)
pd.DataFrame({"missing": missing, "percent": pct})[missing > 0]

,missing,percent
Age,177,19.9
Cabin,687,77.1
Embarked,2,0.2


**Strategy**
- **Age** (about 20% missing): fill with the **median age of the passenger's Pclass and Sex group**. This is more accurate than one global median.
- **Embarked** (2 missing): fill with the **mode** (most common port).
- **Cabin** (about 77% missing): too sparse to fill. Convert into a useful flag `HasCabin`, then drop the column.

In [7]:
clean = df.copy()

# Age: group-wise median imputation
clean["Age"] = clean["Age"].fillna(clean.groupby(["Pclass", "Sex"])["Age"].transform("median"))

# Embarked: mode
clean["Embarked"] = clean["Embarked"].fillna(clean["Embarked"].mode()[0])

# Cabin: keep information as a flag, then drop
clean["HasCabin"] = clean["Cabin"].notna().astype(int)
clean = clean.drop(columns=["Cabin"])

print("Missing values after cleaning:", int(clean.isna().sum().sum()))

Missing values after cleaning: 0


In [8]:
# Duplicates
print("Duplicate rows (all columns):", clean.duplicated().sum())
print("Duplicate PassengerIds       :", clean["PassengerId"].duplicated().sum())
print("Duplicate Tickets (shared tickets are normal for families):", clean["Ticket"].duplicated().sum())
clean = clean.drop_duplicates()
print("Shape after de-duplication:", clean.shape)

Duplicate rows (all columns): 0
Duplicate PassengerIds       : 0
Duplicate Tickets (shared tickets are normal for families): 210
Shape after de-duplication: (891, 12)


## 4. Feature engineering

In [9]:
# Title from Name (e.g. "Braund, Mr. Owen Harris" -> "Mr")
clean["Title"] = clean["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
rare = clean["Title"].value_counts()[lambda s: s < 10].index
clean["Title"] = clean["Title"].replace(rare, "Rare")

# Family size
clean["FamilySize"] = clean["SibSp"] + clean["Parch"] + 1
clean["IsAlone"] = (clean["FamilySize"] == 1).astype(int)

# Age group
clean["AgeGroup"] = pd.cut(clean["Age"], bins=[0, 12, 18, 35, 60, 100],
                           labels=["Child (0-12)", "Teen (13-18)", "Adult (19-35)", "Middle (36-60)", "Senior (60+)"])

clean[["Name", "Title", "FamilySize", "IsAlone", "Age", "AgeGroup"]].head()

,Name,Title,FamilySize,IsAlone,Age,AgeGroup
0,"Braund, Mr. Owen Harris",Mr,2,0,22.0,Adult (19-35)
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs,2,0,38.0,Middle (36-60)
2,"Heikkinen, Miss. Laina",Miss,1,1,26.0,Adult (19-35)
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs,2,0,35.0,Adult (19-35)
4,"Allen, Mr. William Henry",Mr,1,1,35.0,Adult (19-35)


## 5. Grouping and aggregation analysis

In [10]:
overall = clean["Survived"].mean()
print(f"Overall survival rate: {overall:.1%}  ({clean['Survived'].sum()} of {len(clean)} passengers)")

Overall survival rate: 38.4%  (342 of 891 passengers)


In [11]:
# Survival by sex
(clean.groupby("Sex")["Survived"].agg(survival_rate="mean", passengers="count").round(3))

,survival_rate,passengers
Sex,,
female,0.742,314
male,0.189,577


In [12]:
# Survival by passenger class
(clean.groupby("Pclass")["Survived"].agg(survival_rate="mean", passengers="count").round(3))

,survival_rate,passengers
Pclass,,
1,0.630,216
2,0.473,184
3,0.242,491


In [13]:
# Sex x Class (pivot table)
clean.pivot_table(values="Survived", index="Pclass", columns="Sex", aggfunc="mean").round(3)

Sex,female,male
Pclass,,
1,0.968,0.369
2,0.921,0.157
3,0.500,0.135


In [14]:
# Survival by age group, embarkation port, family size
for col in ["AgeGroup", "Embarked", "FamilySize"]:
    print(f"--- {col} ---")
    print(clean.groupby(col, observed=True)["Survived"].agg(survival_rate="mean", passengers="count").round(3), "\n")

--- AgeGroup ---
                survival_rate  passengers
AgeGroup                                 
Child (0-12)            0.580          69
Teen (13-18)            0.429          70
Adult (19-35)           0.358         514
Middle (36-60)          0.384         216
Senior (60+)            0.227          22 

--- Embarked ---
          survival_rate  passengers
Embarked                           
C                 0.554         168
Q                 0.390          77
S                 0.339         646 

--- FamilySize ---


            survival_rate  passengers
FamilySize                           
1                   0.304         537
2                   0.553         161
3                   0.578         102
4                   0.724          29
5                   0.200          15
6                   0.136          22
7                   0.333          12
8                   0.000           6
11                  0.000           7 



In [15]:
# Survival by title
clean.groupby("Title")["Survived"].agg(survival_rate="mean", passengers="count").sort_values("survival_rate", ascending=False).round(3)

,survival_rate,passengers
Title,,
Mrs,0.792,125
Miss,0.698,182
Master,0.575,40
Rare,0.444,27
Mr,0.157,517


In [16]:
# Multiple aggregations on fare and age
clean.groupby("Pclass").agg(
    passengers=("PassengerId", "count"),
    avg_age=("Age", "mean"),
    median_fare=("Fare", "median"),
    max_fare=("Fare", "max"),
    survival_rate=("Survived", "mean"),
).round(2)

,passengers,avg_age,median_fare,max_fare,survival_rate
Pclass,,,,,
1,216,38.27,60.29,512.33,0.63
2,184,29.86,14.25,73.50,0.47
3,491,24.80,8.05,69.55,0.24


## 6. Correlations with survival

In [17]:
num = clean[["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize", "HasCabin"]]
num.corr()["Survived"].drop("Survived").sort_values(ascending=False).round(3)

HasCabin      0.317
Fare          0.257
Parch         0.082
FamilySize    0.017
SibSp        -0.035
Age          -0.060
Pclass       -0.338
Name: Survived, dtype: float64

## 7. Key insights

1. **Overall survival was 38.4%** (342 of 891 passengers).
2. **Sex was the strongest factor.** About **74% of women survived versus 19% of men**.
3. **Class mattered a lot.** Survival fell from **63% (1st class)** to **47% (2nd)** to **24% (3rd)**.
4. **Sex and class together:** about **97% of 1st-class women** survived, but only **14% of 3rd-class men**.
5. **Children had better odds:** about **58% of children (0-12)** survived versus 23% of seniors (60+).
6. **Small families did better than solo travellers or large families.** Groups of 2-4 survived at 55-72%, travellers alone at about 30%, and families of 5 or more mostly at 0-33%.
7. **Port of embarkation:** Cherbourg passengers survived most (about 55%), partly because that group had a higher share of 1st-class tickets.
8. **Having a recorded cabin** (usually upper decks and higher classes) is positively related to survival, and **fare** shows a moderate positive correlation (about +0.26).

**Data-quality notes**
- No fully duplicated rows were found.
- Age was imputed with group medians, so age-based results are approximate for the 177 filled values.
- Correlation does not mean causation. Class, sex and age overlap with each other.

**Skills demonstrated:** DataFrame inspection, missing-value handling, duplicate checks, feature engineering, `groupby`, `pivot_table` and aggregation.

In [18]:
# Save the cleaned dataset for Task 2
clean.to_csv("titanic_cleaned.csv", index=False)
print("Saved titanic_cleaned.csv", clean.shape)

Saved titanic_cleaned.csv (891, 16)
